# Lab 7: Spark Structured Streaming — okna czasowe

## Cel

- Uruchomienie strumienia danych w Spark,
- Źródła danych: rate, file, socket,
- Tryby wyjścia: append, update, complete,
- Agregacje i okna czasowe na strumieniach.

---

## Funkcja pomocnicza

Użyjemy jej do wyświetlania wyników strumienia w notatniku (bez niej strumień działa w nieskończoność).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab7-Streaming").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

batch_counter = {"count": 0}

def process_batch(df, batch_id, max_batches=5):
    batch_counter["count"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(truncate=False)
    if batch_counter["count"] >= max_batches:
        raise Exception("Limit batchów osiągnięty")

## Część 1: Źródło `rate`

### Zadanie 1.1 — Pierwszy strumień

Utwórz strumień z źródła `rate` (1 wiersz/sekundę) i wyświetl go.

In [ ]:
df_rate = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

# Wyswietl schemat
df_rate.printSchema()

# Uruchom strumien
query = (df_rate.writeStream
    .format("console")
    .outputMode("append")
    .foreachBatch(process_batch)
    .start()
)

try:
    query.awaitTermination()
except:
    query.stop()

### Zadanie 1.2 — Wzbogacenie strumienia

Do strumienia `rate` dodaj kolumny:
- `user_id`: `concat('u', cast(rand()*100 as int))`
- `event_type`: `case when rand() > 0.7 then 'purchase' else 'view' end`
- `price`: `round(rand() * 500, 2)`

In [ ]:
from pyspark.sql.functions import expr

events = (df_rate
    # TWÓJ KOD
    # .withColumn("user_id", expr(...))
    # .withColumn("event_type", expr(...))
    # .withColumn("price", expr(...))
)

# Uruchom i sprawdz
batch_counter["count"] = 0
query = (events.writeStream
    .format("console")
    .outputMode("append")
    .foreachBatch(process_batch)
    .start())

try:
    query.awaitTermination()
except:
    query.stop()

### Zadanie 1.3 — Filtrowanie (append mode)

Wyfiltruj tylko zdarzenia `purchase` ze strumienia.

In [ ]:
# TWÓJ KOD
# purchases = events.filter(...)
# Uruchom strumien w trybie append

---

## Część 2: Agregacje na strumieniu

### Zadanie 2.1 — Zliczanie zdarzeń (complete mode)

Policz liczbę zdarzeń per `event_type`.

In [ ]:
# TWÓJ KOD
# agg = events.groupBy("event_type").count()
# Uruchom w outputMode("complete")

### Zadanie 2.2 — Tumbling Window

Pogrupuj zdarzenia w okna 10-sekundowe i policz liczbę zdarzeń + sumę price per okno.

Dodaj watermark 30 sekund.

In [ ]:
from pyspark.sql.functions import window, sum as spark_sum, count

# TWÓJ KOD
# windowed = (events
#     .withWatermark("timestamp", "30 seconds")
#     .groupBy(window("timestamp", "10 seconds"))
#     .agg(count("*").alias("liczba"), spark_sum("price").alias("suma"))
# )
# Uruchom w outputMode("append")

### Zadanie 2.3 — Sliding Window

Zmień okno na sliding: 30 sekund szerokości, krok 10 sekund.

In [ ]:
# TWÓJ KOD

---

## Część 3: Źródło plikowe

### Zadanie 3.1 — Generator plików JSON

Uruchom poniższy generator w osobnym terminalu (`python generator.py`):

In [ ]:
%%file generator.py
import json, os, random, time
from datetime import datetime, timedelta

os.makedirs("data/stream", exist_ok=True)

def generate_event():
    return {
        "user_id": f"u{random.randint(1, 50)}",
        "event_type": random.choices(["view", "cart", "purchase"], weights=[0.6, 0.25, 0.15])[0],
        "timestamp": datetime.utcnow().isoformat(),
        "product_id": f"p{random.randint(100, 120)}",
        "price": round(random.uniform(10, 1000), 2)
    }

while True:
    batch = [generate_event() for _ in range(20)]
    fname = f"data/stream/events_{int(time.time())}.json"
    with open(fname, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {fname}")
    time.sleep(5)

### Zadanie 3.2 — Odczyt strumienia z katalogu

Zdefiniuj schemat danych i uruchom strumień czytający z `data/stream/`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# TWÓJ KOD
# schema = StructType([...])
# stream = spark.readStream.schema(schema).json("data/stream")
# Uruchom i wyswietl

---

## Praca domowa

1. Na źródle plikowym wykonaj segmentację klientów:
   - `purchase` w oknie → "Buyer"
   - `cart` bez `purchase` → "Cart abandoner"
   - tylko `view` → "Lurker"
   Podpowiedź: `collect_set("event_type")` + `array_contains()`
2. Wyślij kod do repozytorium Git.

Na następnych zajęciach: Spark + Kafka — integracja.

In [ ]:
spark.stop()